<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자연어 처리와 LLM · 06. T5

## Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer

- **원 논문:** [Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer](https://arxiv.org/abs/1910.10683)
- **저자 / 발표:** Colin Raffel et al. · JMLR 2020 (2019)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch, 확률·선형대수·sequence model의 기초
- **로컬 학습 데이터:** `data/field_curriculum/nlp_corpus.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** text-to-text 형식, span sentinel target, relative position bias를 재현한다.

**축소하거나 생략한 부분:** C4와 최대 11B parameter 대신 60개 3-token sequence와 작은 Transformer를 쓰며, 48개 train/12개 held-out 행을 분리한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1.4 and Fig. 2: span replacement with sentinel tokens | Task 1 | sentinel input/target sequence |
| §2.1: encoder-decoder attention and relative position embeddings | Task 2 explicit encoder/causal decoder/cross-attention | bias shape, nonzero gradient, and future invariance |
| §3.3.4: span-corruption objective | Task 3 | denoising accuracy |

## 핵심 아이디어와 수식

$$L_{span}=-\sum_t\log p(y_t\mid y_{<t},\widetilde{x})$$

**기호 해설:** x-tilde는 sentinel로 span을 바꾼 입력, y는 sentinel과 제거 span을 잇는 target이다.

**코드 대응:** Task 1이 span pair, Task 2가 logits에 relative bias를 더하는 encoder/causal decoder/cross-attention, Task 3이 denoising CE를 맡는다.

**입력과 출력 shape:** train source [48,2], decoder [48,4]와 held-out source [12,2]를 분리해 logits [B,4,V]와 token accuracy를 계산한다.

**포트폴리오 구현 범위:** 여러 span과 bucketed bias 대신 한 span/clip distance를 쓰되 text-to-text 목적은 유지한다.

각 단계는 데이터 전처리와 shape 확인부터 논문 고유 class, `forward`, 목적함수,
`train_step`, 평가까지 이어진다. 핵심 수식을 한 번의 고수준 호출로 감추지 않고
중간 tensor를 확인할 수 있게 분리한다.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import json
import math
import re
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/nlp_corpus.json")
corpus = json.loads(dataset_path.read_text(encoding="utf-8"))
sentences = corpus["sentences"]
translation_pairs = corpus["translation_pairs"]
documents = corpus["documents"]
queries = corpus["queries"]
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<mask>", "<sentinel>"]
sentence_words = {word.lower() for sentence in sentences for word in sentence.split()}
translation_words = {
    word.lower()
    for pair in translation_pairs
    for side in ("source", "target")
    for word in pair[side].split()
}
words = sorted(sentence_words | translation_words)
itos = SPECIAL_TOKENS + words
stoi = {word: index for index, word in enumerate(itos)}
PAD, BOS, EOS, MASK, SENTINEL = range(5)
VOCAB_SIZE = len(itos)


def encode(text, length=4):
    token_ids = [stoi[word.lower()] for word in text.split()]
    token_ids = token_ids[:length]
    return token_ids + [PAD] * (length - len(token_ids))


sentence_ids = torch.tensor([encode(text) for text in sentences])
sentiment_labels = torch.tensor(corpus["sentiment_labels"], dtype=torch.long)
source_ids = torch.tensor([encode(pair["source"], 3) for pair in translation_pairs])
target_ids = torch.tensor([encode(pair["target"], 3) for pair in translation_pairs])
sentence_ids, sentiment_labels = ACCELERATOR.move(
    sentence_ids,
    sentiment_labels,
)
source_ids, target_ids = ACCELERATOR.move(source_ids, target_ids)
assert sentence_ids.shape == (360, 4)
assert source_ids.shape == target_ids.shape == (60, 3)
print(ACCELERATOR.summary())
print(dataset_path.name, f"sentences={len(sentences)}, vocab={VOCAB_SIZE}")

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1.4 and Fig. 2: span replacement with sentinel tokens
- **노트북에서 구현할 부분:** Task 1
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** sentinel input/target sequence

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
@dataclass
class T5Config:
    model_dim: int = 24
    heads: int = 4
    feedforward_dim: int = 48
    max_distance: int = 4
    learning_rate: float = 0.02
    steps: int = 32


def make_span_corruption(source, sentinel_id, eos_id, bos_id):
    sentinel = source.new_full((len(source),), sentinel_id)
    eos = source.new_full((len(source),), eos_id)
    corrupted = torch.stack((source[:, 0], sentinel), dim=1)
    target = torch.stack(
        (sentinel, source[:, 1], source[:, 2], eos),
        dim=1,
    )
    decoder_inputs = torch.cat(
        (source.new_full((len(source), 1), bos_id), target[:, :-1]),
        dim=1,
    )
    return corrupted, target, decoder_inputs


config = T5Config()
split_generator = torch.Generator().manual_seed(32)
translation_order = torch.randperm(
    len(source_ids),
    generator=split_generator,
).to(source_ids.device)
train_translation_index = translation_order[:48]
test_translation_index = translation_order[48:]
assert not bool(
    torch.isin(
        train_translation_index,
        test_translation_index,
    ).any()
)
train_span_input, train_span_target, train_decoder_inputs = make_span_corruption(
    source_ids[train_translation_index],
    SENTINEL,
    EOS,
    BOS,
)
test_span_input, test_span_target, test_decoder_inputs = make_span_corruption(
    source_ids[test_translation_index],
    SENTINEL,
    EOS,
    BOS,
)
assert train_span_input.shape == (48, 2)
assert test_span_input.shape == (12, 2)
assert train_span_target.shape == train_decoder_inputs.shape == (48, 4)
assert test_span_target.shape == test_decoder_inputs.shape == (12, 4)

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2.1: encoder-decoder attention and relative position embeddings
- **노트북에서 구현할 부분:** Task 2 explicit encoder/causal decoder/cross-attention
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** bias shape, nonzero gradient, and future invariance

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
class RelativePositionBias(nn.Module):
    """Clipped relative distance를 head별 attention bias로 바꾼다."""

    def __init__(self, heads, max_distance):
        super().__init__()
        self.max_distance = max_distance
        self.embedding = nn.Embedding(2 * max_distance + 1, heads)

    def forward(self, query_length, key_length, device):
        query = torch.arange(query_length, device=device)[:, None]
        key = torch.arange(key_length, device=device)[None, :]
        relative = (key - query).clamp(
            -self.max_distance,
            self.max_distance,
        )
        indices = relative + self.max_distance
        return self.embedding(indices).permute(2, 0, 1)


class ExplicitMultiHeadAttention(nn.Module):
    """QK^T/sqrt(d) 위에 bias와 mask를 직접 적용한다."""

    def __init__(self, model_dim, heads):
        super().__init__()
        if model_dim % heads != 0:
            raise ValueError("model_dim must be divisible by heads")
        self.heads = heads
        self.head_dim = model_dim // heads
        self.query_projection = nn.Linear(model_dim, model_dim)
        self.key_projection = nn.Linear(model_dim, model_dim)
        self.value_projection = nn.Linear(model_dim, model_dim)
        self.output_projection = nn.Linear(model_dim, model_dim)

    def split_heads(self, tensor):
        batch, length, _ = tensor.shape
        tensor = tensor.reshape(
            batch,
            length,
            self.heads,
            self.head_dim,
        )
        return tensor.transpose(1, 2)

    def merge_heads(self, tensor):
        batch, _, length, _ = tensor.shape
        tensor = tensor.transpose(1, 2).contiguous()
        return tensor.reshape(
            batch,
            length,
            self.heads * self.head_dim,
        )

    def forward(
        self,
        query,
        key,
        value,
        relative_bias=None,
        attention_mask=None,
    ):
        queries = self.split_heads(self.query_projection(query))
        keys = self.split_heads(self.key_projection(key))
        values = self.split_heads(self.value_projection(value))
        scale = self.head_dim**-0.5
        scores = torch.matmul(queries, keys.transpose(-2, -1)) * scale
        if relative_bias is not None:
            scores = scores + relative_bias[None]
        if attention_mask is not None:
            scores = scores.masked_fill(
                attention_mask[None, None],
                float("-inf"),
            )
        weights = scores.softmax(dim=-1)
        context = torch.matmul(weights, values)
        output = self.output_projection(self.merge_heads(context))
        return output, weights


class T5EncoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.model_dim)
        self.attention = ExplicitMultiHeadAttention(
            config.model_dim,
            config.heads,
        )
        self.norm2 = nn.LayerNorm(config.model_dim)
        self.feedforward = nn.Sequential(
            nn.Linear(config.model_dim, config.feedforward_dim),
            nn.ReLU(),
            nn.Linear(config.feedforward_dim, config.model_dim),
        )

    def forward(self, hidden, relative_bias):
        normalized = self.norm1(hidden)
        attended, weights = self.attention(
            normalized,
            normalized,
            normalized,
            relative_bias=relative_bias,
        )
        hidden = hidden + attended
        hidden = hidden + self.feedforward(self.norm2(hidden))
        return hidden, weights


class T5DecoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.model_dim)
        self.self_attention = ExplicitMultiHeadAttention(
            config.model_dim,
            config.heads,
        )
        self.norm2 = nn.LayerNorm(config.model_dim)
        self.cross_attention = ExplicitMultiHeadAttention(
            config.model_dim,
            config.heads,
        )
        self.norm3 = nn.LayerNorm(config.model_dim)
        self.feedforward = nn.Sequential(
            nn.Linear(config.model_dim, config.feedforward_dim),
            nn.ReLU(),
            nn.Linear(config.feedforward_dim, config.model_dim),
        )

    def forward(self, hidden, memory, relative_bias, causal_mask):
        normalized = self.norm1(hidden)
        attended, self_weights = self.self_attention(
            normalized,
            normalized,
            normalized,
            relative_bias=relative_bias,
            attention_mask=causal_mask,
        )
        hidden = hidden + attended
        normalized = self.norm2(hidden)
        attended, cross_weights = self.cross_attention(
            normalized,
            memory,
            memory,
        )
        hidden = hidden + attended
        hidden = hidden + self.feedforward(self.norm3(hidden))
        return hidden, self_weights, cross_weights


class T5Mini(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config.model_dim)
        self.encoder_relative_bias = RelativePositionBias(
            config.heads,
            config.max_distance,
        )
        self.decoder_relative_bias = RelativePositionBias(
            config.heads,
            config.max_distance,
        )
        self.encoder = T5EncoderBlock(config)
        self.decoder = T5DecoderBlock(config)
        self.output = nn.Linear(config.model_dim, vocab_size)

    def causal_mask(self, length, device):
        return torch.ones(
            length,
            length,
            dtype=torch.bool,
            device=device,
        ).triu(diagonal=1)

    def forward(self, source, decoder_inputs):
        source_hidden = self.embedding(source)
        source_bias = self.encoder_relative_bias(
            source.shape[1],
            source.shape[1],
            source.device,
        )
        memory, _ = self.encoder(source_hidden, source_bias)
        target_hidden = self.embedding(decoder_inputs)
        target_length = decoder_inputs.shape[1]
        target_bias = self.decoder_relative_bias(
            target_length,
            target_length,
            source.device,
        )
        target_mask = self.causal_mask(target_length, source.device)
        decoded, _, _ = self.decoder(
            target_hidden,
            memory,
            target_bias,
            target_mask,
        )
        return self.output(decoded)


model = T5Mini(VOCAB_SIZE, config).to(DEVICE)
bias = model.encoder_relative_bias(2, 4, DEVICE)
assert bias.shape == (4, 2, 4)
train_logits = model(train_span_input, train_decoder_inputs)
assert train_logits.shape == (48, 4, VOCAB_SIZE)
probe = train_logits[0, -1, SENTINEL]
probe.backward()
encoder_gradient = model.encoder_relative_bias.embedding.weight.grad
decoder_gradient = model.decoder_relative_bias.embedding.weight.grad
assert encoder_gradient is not None
assert decoder_gradient is not None
assert float(encoder_gradient.abs().sum()) > 0.0
assert float(decoder_gradient.abs().sum()) > 0.0
model.zero_grad(set_to_none=True)

changed_inputs = train_decoder_inputs[:2].clone()
changed_inputs[:, -1] = EOS
with torch.no_grad():
    original_logits = model(
        train_span_input[:2],
        train_decoder_inputs[:2],
    )
    changed_logits = model(train_span_input[:2], changed_inputs)
assert torch.allclose(
    original_logits[:, :-1],
    changed_logits[:, :-1],
    atol=1e-6,
)

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.3.4: span-corruption objective
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** denoising accuracy

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
def denoising_loss(logits, targets):
    return F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
    )


def train_t5_step(model, optimizer, source, decoder_inputs, targets):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = denoising_loss(model(source, decoder_inputs), targets)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def evaluate_t5(model, source, decoder_inputs, targets):
    model.eval()
    with torch.no_grad():
        predictions = model(source, decoder_inputs).argmax(dim=-1)
    return float((predictions == targets).float().mean())


optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
losses = []
for step in range(config.steps):
    losses.append(
        train_t5_step(
            model,
            optimizer,
            train_span_input,
            train_decoder_inputs,
            train_span_target,
        )
    )
heldout_accuracy = evaluate_t5(
    model,
    test_span_input,
    test_decoder_inputs,
    test_span_target,
)
assert min(losses[3:]) < losses[0]
assert heldout_accuracy > 0.75
plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.title(f"T5 held-out span accuracy={heldout_accuracy:.1%}")
plt.tight_layout()
plt.show()

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> C4와 최대 11B parameter 대신 60개 3-token sequence와 작은 Transformer를 쓰며, 48개 train/12개 held-out 행을 분리한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.